In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
# DN = 'C://work/dev/python/progs/texts/sec_bert/'
DN = '/home/jovyan/work/sec_bert/'
os.chdir(DN)

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt
import seaborn as sns

import os

from ruamel.yaml import YAML
import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain

import click
import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')
from src.funcs import set_seed
from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df
from src.spec_nn_funcs import TextDFDataset, TextModelClass


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import RobustScaler

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from ruamel.yaml import YAML

conf = YAML().load(open('params.yaml'))
conf_ttp = YAML().load(open('dvc_pipes/ttp/params_ttp.yaml'))
conf_bert = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))
conf_bert_ttp = YAML().load(open('dvc_pipes/bert_ttp/params_bert_ttp.yaml'))

set_seed(conf['seed'])

In [ ]:
mlb_ttp = joblib.load(conf['prep_text']['ttp_mlb_fn'])
mlb = joblib.load(conf['prep_text']['mlb_fn'])

data_ttp = pd.read_csv(conf_ttp['feat_gen_ttp']['data_fn'])
data_ttp['target'] = data_ttp['target'].map(lambda x: eval(x))
data_ttp['ttp'] = data_ttp['ttp'].map(lambda x: eval(x))
data_ttp['labels'] = data_ttp['labels'].map(lambda x: eval(x))


data = pd.read_csv(conf['feat_gen']['data_fn'])
data['target'] = data['target'].map(lambda x: eval(x))
data['labels'] = data['labels'].map(lambda x: eval(x))

tr_idx = data.query('split=="tr"').index
val_idx = data.query('split=="val"').index
ts_idx = data.query('split=="ts"').index

In [ ]:
tr_ttp_idx = data_ttp.query('split=="tr"').index
val_ttp_idx = data_ttp.query('split=="val"').index
ts_ttp_idx = data_ttp.query('split=="ts"').index

In [ ]:
bert_type = conf_bert['nn_bert']['bert_type']


In [ ]:
VALID_BATCH_SIZE = conf_bert['nn']['batch_size']
TRAIN_BATCH_SIZE = conf_bert['nn']['batch_size']
MAX_SEQ_LENGTH = conf_bert['nn']['maxlen']

checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}


# Предсказания тактик

In [ ]:
tr_ds = TextDFDataset(data.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ds = TextDFDataset(data.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ds = TextDFDataset(data.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ld = DataLoader(tr_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ld = DataLoader(val_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ld = DataLoader(ts_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
name = os.path.basename(conf_bert['nn_bert']['model_fn'])
dirname = os.path.dirname(conf_bert['nn_bert']['model_fn'])

# model_bert = torch.load(f'{dirname}/{bert_type}_{name}')
model_bert = torch.load(conf_bert['nn_bert']['model_fn'])

Y_val_proba = np.array(get_preds(model_bert, ld=val_ld)['pred'])
Y_tr_proba = np.array(get_preds(model_bert, ld=tr_ld)['pred'])
Y_ts_proba = np.array(get_preds(model_bert, ld=ts_ld)['pred'])


In [ ]:
thresh_l = get_opt_thresh(y_true = np.array(data.loc[val_idx, 'target'].values.tolist()), 
                          probas = Y_val_proba, mlb = mlb, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=[], dump_fn = None)

In [ ]:
nttp_df = pd.concat([data[['sentence', 'labels', 'target', 'split']].query('split=="tr"').assign(proba_labels = Y_tr_proba.tolist()),
          data[['sentence', 'labels', 'target', 'split']].query('split=="val"').assign(proba_labels = Y_val_proba.tolist()),
           data[['sentence', 'labels', 'target', 'split']].query('split=="ts"').assign(proba_labels = Y_ts_proba.tolist())
          ], axis=0, ignore_index=True)

nttp_df['pred_labels'] = nttp_df['proba_labels'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_l)])

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(nttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(nttp_df.query('split=="val"')['pred_labels'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(nttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(nttp_df.query('split=="val"')['pred_labels'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

# Предсказания техник

In [ ]:
tr_ttp_ds = TextDFDataset(data_ttp.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ttp_ds = TextDFDataset(data_ttp.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ttp_ds = TextDFDataset(data_ttp.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ttp_ld = DataLoader(tr_ttp_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ttp_ld = DataLoader(val_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ttp_ld = DataLoader(ts_ttp_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))


In [ ]:
name = os.path.basename(conf_bert_ttp['nn_bert_ttp']['model_fn'])
dirname = os.path.dirname(conf_bert_ttp['nn_bert_ttp']['model_fn'])

# model_bert_ttp = torch.load(f'{dirname}/{bert_type}_{name}')
model_bert_ttp = torch.load(conf_bert_ttp['nn_bert_ttp']['model_fn'])

Y_val_proba = np.array(get_preds(model_bert_ttp, ld=val_ttp_ld)['pred'])
Y_tr_proba = np.array(get_preds(model_bert_ttp, ld=tr_ttp_ld)['pred'])
Y_ts_proba = np.array(get_preds(model_bert_ttp, ld=ts_ttp_ld)['pred'])

In [ ]:
# thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[val_ttp_idx, 'target'].values.tolist()), 
#                           probas = Y_val_proba, mlb = mlb_ttp, 
#                           opt_metric=conf['train_eval_model']['opt_metric'], 
#                           thresh_space_l=np.arange(0.005, 1, 0.005), dump_fn = None)

thresh_ttp_l = get_opt_thresh(y_true = np.array(data_ttp.loc[data_ttp.split=='tr', 'target'].values.tolist()), 
                          probas = Y_tr_proba, mlb = mlb_ttp, 
                          opt_metric=conf['train_eval_model']['opt_metric'], 
                          thresh_space_l=np.arange(0.001, 1, 0.002), dump_fn = None)

In [ ]:
ttp_df = pd.concat([data_ttp[['sentence', 'ttp', 'target', 'split']].query('split=="tr"').assign(proba_ttp = Y_tr_proba.tolist()),
          data_ttp[['sentence', 'ttp', 'target', 'split']].query('split=="val"').assign(proba_ttp = Y_val_proba.tolist()),
           data_ttp[['sentence', 'ttp', 'target', 'split']].query('split=="ts"').assign(proba_ttp = Y_ts_proba.tolist())
          ], axis=0, ignore_index=True)

ttp_df['pred_ttp'] = ttp_df['proba_ttp'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_ttp_l)])

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(ttp_df.query('split=="val"')['target'].values.tolist()), 
                                                    np.array(ttp_df.query('split=="val"')['pred_ttp'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

# Объединяем

In [ ]:
# у некоторых предложений маскировка может заменить имя название малвари и замены будут одинаковые (например, на It
# так было тут - data[data.sentence.str.contains('has the ability to use FTP in C2')]
df = nttp_df.rename(columns={'target':'enc_labels'}).merge(
    ttp_df.rename(columns={'target':'enc_ttp'}), on=['sentence', 'split'], how='left')\
    .reset_index(drop=True)

# df = data[['sentence', 'labels', 'target', 'split']].rename(columns={'target':'y_labels'}).merge(
#     data_ttp.drop_duplicates(subset=['sentence'])[['sentence', 'ttp', 'target', 'split']].rename(columns={'target':'y_ttp'}),
#     on=['sentence', 'split'], how='left')
df.shape

In [ ]:
df['pred_str_labels'] = df['pred_labels'].map(lambda x: mlb.inverse_transform(np.array([x]))[0])

df['pred_str_ttp'] = df['pred_ttp'].map(lambda x: mlb_ttp.inverse_transform(np.array([x]))[0])

# генетика

### для всех фич

In [ ]:
%%time
set_seed(conf['seed'])
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from src.meta import BFSThres, fake_splitter, get_toolbox, get_gen_proba
from src.funcs import get_pred_thresh

from sklearn.metrics import average_precision_score, f1_score
from deap import tools
from deap import algorithms

# model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
#                             class_weight='balanced', penalty=None)

model = DecisionTreeClassifier(class_weight='balanced', max_depth=2, random_state=conf['seed'])

average_precision_score_l = []
average_precision_score_bert_l = []
f1_score_l = []
f1_score_bert_l = []

average_precision_score_val_l = []
average_precision_score_val_bert_l = []
f1_score_val_l = []
f1_score_bert_val_l = []
f_l = []
feat_l = []
average_precision_score_tr_l = []

N_GEN = 20

from time import time

st = time()

for i in tqdm(range(len(mlb_ttp.classes_))):
    
    X = df.apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1)

    X = pd.DataFrame(X.values.tolist(), columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    
    y = df['enc_ttp'].map(lambda x: x[i])
    
    selector = BFSThres(model, scoring='average_precision', cv = fake_splitter(df), thresh=0.1, metric_sign=1)
    selector.fit(X, y, verbose=True)
    
    filt_cols = np.array(selector.mask)
    filt_cols = X.columns[filt_cols]
    # if len(filt_cols)==1:
    #     filt_cols = ['ttp_i']
    
    X_tr = pd.DataFrame(df.query('split=="tr"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_tr = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])
    
    X_val = pd.DataFrame(df.query('split=="val"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i])
    
    X_ts = pd.DataFrame(df.query('split=="ts"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_ts = df.query('split=="ts"')['enc_ttp'].map(lambda x: x[i])


    feat_l.append(filt_cols)
    
    N = len(filt_cols)
    
    toolbox = get_toolbox(N=N)

    def evalSymbReg(individual, points, y_true):
        # Transform the tree expression in a callable function
        f = toolbox.compile(expr=individual)
        y_pred=[f(*point) for point in points]
        # обрежем значения
        y_pred=[min(0.99, it) for it in y_pred]
        y_pred=[max(0, it) for it in y_pred]
        
        loss = log_loss(y_true=y_true, y_pred=y_pred)
        
        return loss,

    
    # toolbox.register("evaluate", evalSymbReg, points=X_tr[filt_cols].values.tolist(), y_true=y_tr)
    toolbox.register("evaluate", evalSymbReg, points=X_val[filt_cols].values.tolist(), y_true=y_val)

    
    hof = tools.HallOfFame(1)
    
    population, logbook = algorithms.eaSimple(toolbox.population(n=100), toolbox, 
                                              cxpb=0.5, mutpb=0.2, ngen=N_GEN, halloffame=hof, verbose=False)
    
    f = toolbox.compile(hof[0])
    f_l.append(str(hof[0]))
    
    y_tr_proba = get_gen_proba(f, X_tr, filt_cols)
    y_val_proba = get_gen_proba(f, X_val, filt_cols)
    y_ts_proba = get_gen_proba(f, X_ts, filt_cols)

    
    average_precision_score_l.append(average_precision_score(y_ts, y_ts_proba))
    average_precision_score_val_l.append(average_precision_score(y_val, y_val_proba))
    average_precision_score_tr_l.append(average_precision_score(y_tr, y_tr_proba))
    
    _, thresh_i = get_pred_thresh(y_true = y_tr, proba=y_tr_proba, opt_metric=conf['train_eval_model']['opt_metric'],
                               thresh_space_l=np.arange(0.001, 1, 0.002))


    f1_score_l.append(f1_score(y_ts, (y_ts_proba>thresh_i)))
    f1_score_val_l.append(f1_score(y_val, (y_val_proba>thresh_i)))
    
    average_precision_score_bert_l.append(average_precision_score(y_ts, df.query('split=="ts"')['proba_ttp'].map(lambda x: x[i])))
    average_precision_score_val_bert_l.append(average_precision_score(y_val, df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])))
    
    f1_score_bert_l.append(f1_score(y_ts, df.query('split=="ts"')['pred_ttp'].map(lambda x: x[i])))
    f1_score_bert_val_l.append(f1_score(y_val, df.query('split=="val"')['pred_ttp'].map(lambda x: x[i])))

    
end = time()

In [ ]:
(end-st)/3600

In [ ]:
np.mean(average_precision_score_tr_l), np.mean(average_precision_score_val_l), np.mean(average_precision_score_val_bert_l), np.mean(f1_score_val_l), np.mean(f1_score_bert_val_l)

In [ ]:
thresh = 0.2
pd.DataFrame({'bert':average_precision_score_val_bert_l, 'gen':average_precision_score_val_l}).query('bert<@thresh')\
            .assign(gen_good=lambda x:x['gen']>x['bert'])\
            ['gen_good'].mean()

In [ ]:
np.mean(average_precision_score_l), np.mean(average_precision_score_bert_l), np.mean(f1_score_l), np.mean(f1_score_bert_l)

### генетика только для нескольких фич

In [ ]:
%%time

set_seed(conf['seed'])
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from src.meta import BFSThres, fake_splitter, get_toolbox, get_gen_proba
from src.funcs import get_pred_thresh

from sklearn.metrics import average_precision_score, f1_score
from deap import tools
from deap import algorithms

# model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
#                             class_weight='balanced', penalty=None)

model = DecisionTreeClassifier(class_weight='balanced', max_depth=2, random_state=conf['seed'])

average_precision_score_l = []
average_precision_score_bert_l = []
f1_score_l = []
f1_score_bert_l = []

average_precision_score_tr_l = []

average_precision_score_val_l = []
average_precision_score_val_bert_l = []
f1_score_val_l = []
f1_score_bert_val_l = []
f_l = []
feat_l = []

N_GEN = 50

from time import time

st = time()
tak_col = 'proba_labels' # 'proba_labels'
i_l = []
for i in tqdm(range(len(mlb_ttp.classes_))):
    
    X = df.apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1)

    X = pd.DataFrame(X.values.tolist(), columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    
    y = df['enc_ttp'].map(lambda x: x[i])
    
    selector = BFSThres(model, scoring='average_precision', cv = fake_splitter(df), thresh=0.1, metric_sign=1)
    selector.fit(X, y, verbose=True)
    
    filt_cols = np.array(selector.mask)
    filt_cols = X.columns[filt_cols]
    if not 'ttp_i' in filt_cols:
        # import pdb;pdb.set_trace()
        filt_cols = filt_cols.tolist() + ['ttp_i']
    if len(filt_cols)==1:
        filt_cols = ['ttp_i']
        continue
    i_l.append(i)
    X_tr = pd.DataFrame(df.query('split=="tr"').apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_tr = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])
    
    X_val = pd.DataFrame(df.query('split=="val"').apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i])
    
    X_ts = pd.DataFrame(df.query('split=="ts"').apply(lambda x: x[tak_col] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_ts = df.query('split=="ts"')['enc_ttp'].map(lambda x: x[i])


    feat_l.append(filt_cols)
    
    N = len(filt_cols)
    
    toolbox = get_toolbox(N=N)

    def evalSymbReg(individual, points, y_true):
        # Transform the tree expression in a callable function
        f = toolbox.compile(expr=individual)
        y_pred=[f(*point) for point in points]
        # обрежем значения
        y_pred=[min(0.99, it) for it in y_pred]
        y_pred=[max(0, it) for it in y_pred]
        
        loss = log_loss(y_true=y_true, y_pred=y_pred)
        
        return loss,
        
    # toolbox.register("evaluate", evalSymbReg, points=X_tr[filt_cols].values.tolist(), y_true=y_tr)
    toolbox.register("evaluate", evalSymbReg, points=X_val[filt_cols].values.tolist(), y_true=y_val)
    
    hof = tools.HallOfFame(1)
    
    population, logbook = algorithms.eaSimple(toolbox.population(n=100), toolbox, 
                                              cxpb=0.5, mutpb=0.2, ngen=N_GEN, halloffame=hof, verbose=False)
    
    f = toolbox.compile(hof[0])
    f_l.append(str(hof[0]))
    
    y_tr_proba = get_gen_proba(f, X_tr, filt_cols)
    y_val_proba = get_gen_proba(f, X_val, filt_cols)
    y_ts_proba = get_gen_proba(f, X_ts, filt_cols)


    average_precision_score_tr_l.append(average_precision_score(y_tr, y_tr_proba))
    average_precision_score_l.append(average_precision_score(y_ts, y_ts_proba))
    average_precision_score_val_l.append(average_precision_score(y_val, y_val_proba))
    
    _, thresh_i = get_pred_thresh(y_true = y_tr, proba=y_tr_proba, opt_metric=conf['train_eval_model']['opt_metric'],
                               thresh_space_l=np.arange(0.001, 1, 0.002))


    f1_score_l.append(f1_score(y_ts, (y_ts_proba>thresh_i)))
    f1_score_val_l.append(f1_score(y_val, (y_val_proba>thresh_i)))
    
    average_precision_score_bert_l.append(average_precision_score(y_ts, df.query('split=="ts"')['proba_ttp'].map(lambda x: x[i])))
    average_precision_score_val_bert_l.append(average_precision_score(y_val, df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])))
    
    f1_score_bert_l.append(f1_score(y_ts, df.query('split=="ts"')['pred_ttp'].map(lambda x: x[i])))
    f1_score_bert_val_l.append(f1_score(y_val, df.query('split=="val"')['pred_ttp'].map(lambda x: x[i])))

    
end = time()

In [ ]:
(end-st)/3600

In [ ]:
np.mean(average_precision_score_tr_l), np.mean(average_precision_score_val_l), np.mean(average_precision_score_val_bert_l), np.mean(f1_score_val_l), np.mean(f1_score_bert_val_l)

In [ ]:
np.mean(average_precision_score_l), np.mean(average_precision_score_bert_l), np.mean(f1_score_l), np.mean(f1_score_bert_l)

In [ ]:
comp_ts_df = pd.DataFrame({'bert':average_precision_score_bert_l, 'gen':average_precision_score_l}).assign(diff=lambda x: x['bert'] - x['gen']).sort_values(by='diff')
comp_val_df = pd.DataFrame({'bert':average_precision_score_val_bert_l, 'gen':average_precision_score_val_l}).assign(diff=lambda x: x['bert'] - x['gen']).sort_values(by='diff')

comp_val_df

### сравнение по классам

In [ ]:
comp_val_df.merge(comp_ts_df, left_index=True, right_index=True).query('gen_x>bert_x and gen_y>bert_y')\
            .assign(diff=lambda x: x['diff_x']+ x['diff_y'])\
            .sort_values(by='diff')

#### другой способ

In [ ]:
val_gen_idx = pd.DataFrame({'bert':average_precision_score_val_bert_l, 'gen':average_precision_score_val_l}).query('gen>bert').index
val_gen_idx

In [ ]:
ts_gen_idx = pd.DataFrame({'bert':average_precision_score_bert_l, 'gen':average_precision_score_l}).query('gen>bert').index
ts_gen_idx

In [ ]:
ts_gen_idx.intersection(val_gen_idx)

## i = 27

In [ ]:
i = 27
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
mlb_ttp.classes_[i_l[i]]

In [ ]:
f_l[i]

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]]*x['proba_labels'][8], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==0][['gen','bert']].mean()

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==1][['gen','bert']]

<div class='alert alert-info'> Для 0 лучше прогнозы, ближе к 0 
</div>

## i = 30

In [ ]:
i = 30
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
mlb_ttp.classes_[i_l[i]]

In [ ]:
f_l[i]

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]]*x['proba_labels'][4], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==0][['gen','bert']].mean()

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==1][['gen','bert']]

<div class='alert alert-info'> Для 0 лучше прогнозы, ближе к 0 
</div>

## i = 32

In [ ]:
i = 32
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
mlb_ttp.classes_[i_l[i]]

In [ ]:
f_l[i]

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]]*x['proba_labels'][0], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==0][['gen','bert']].mean()

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==1][['gen','bert']]

<div class='alert alert-info'> Для 0 лучше прогнозы, ближе к 0 
</div>

## i = 69

In [ ]:
i = 69
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
mlb_ttp.classes_[i_l[i]]

In [ ]:
f_l[i]

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]]*x['proba_labels'][8], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==0][['gen','bert']].mean()

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==1][['gen','bert']]

<div class='alert alert-info'> Для 0 лучше прогнозы, ближе к 0 
</div>

## i = 24

In [ ]:
i = 24
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
mlb_ttp.classes_[i_l[i]]

In [ ]:
f_l[i]

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]]*x['proba_labels'][5], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

In [ ]:
df.query('split=="val"').assign(gen=y_val_proba, bert=lambda x: x['proba_ttp'].map(lambda y: y[i_l[i]]))\
    .loc[y_val==0][['gen','bert']].mean()

<div class='alert alert-info'> Для 0 лучше прогнозы, ближе к 0 
</div>

## i = 8

In [ ]:
mlb_ttp.classes_[i_l[8]], mlb.classes_[3]

In [ ]:
i = 8
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
f_l[i]

In [ ]:
ttp_i - tak_3

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[8]])

y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[8]] - x['proba_labels'][3], axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[8]], axis=1))

In [ ]:
pd.DataFrame({'y_p':y_val_proba, 'y_bert':df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[8]], axis=1), 'y':y_val}).loc[lambda x: x['y']==0]

## проверка одного кейса

In [ ]:
i = 9
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
f_l[i]

In [ ]:
ttp_i - (2*tak_12+tak_11)

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i])

y_val_proba = df.query('split=="val"').apply(lambda x: x['proba_ttp'][i] - (2*x['proba_labels'][12] + x['proba_labels'][11]), axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i], axis=1))

In [ ]:
pd.DataFrame({'y_p':y_val_proba, 'y_bert':df.query('split=="val"').apply(lambda x: x['proba_ttp'][i], axis=1), 'y':y_val}).loc[lambda x: x['y']==1]

In [ ]:
df.query('split=="val"').apply(lambda x: x['enc_ttp'][i]==1, axis=1).loc[lambda x: x==True]


In [ ]:
y_val[y_val.index.isin([22732, 22733])], y_val_proba[y_val_proba.index.isin([22732, 22733])]

In [ ]:
sum(y_val_proba>0.32)

In [ ]:
df[df.index.isin([22732, 22733])]

In [ ]:
comp_val_df

## i = 36

In [ ]:
i = 36
pd.DataFrame({'feat':feat_l[i], '': [f'ARG{it}' for it in range(len(feat_l[i]))]})


In [ ]:
mlb_ttp.classes_[i_l[i]]

In [ ]:
f_l[i]

In [ ]:
average_precision_score_val_l[i], average_precision_score_val_bert_l[i], average_precision_score_l[i], average_precision_score_bert_l[i]

In [ ]:
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i_l[i]])
display(y_val.sum())
y_val_proba = df.query('split=="val"').apply(lambda x: 0.0015, axis=1)
y_val_proba=[min(0.99, it) for it in y_val_proba]
y_val_proba=[max(0, it) for it in y_val_proba]


average_precision_score(y_val, y_val_proba), average_precision_score(y_val, df.query('split=="val"').apply(lambda x: x['proba_ttp'][i_l[i]], axis=1))

### manual

In [ ]:
from sklearn.linear_model import LogisticRegression
from src.meta import BFSThres, fake_splitter, get_toolbox, get_proba, evalSymbReg

model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
                            class_weight='balanced', penalty=None)

# for i in range(len(mlb_ttp.classes_)):

i = 36

# X =  df.apply(lambda x: x['proba_labels'] + x['proba_ttp'], axis=1)
X = df.apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1)

X = pd.DataFrame(X.values.tolist(), columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
# X = pd.DataFrame(df['feat'].values.tolist(), columns=[f'tak_{i}' for i in range(14)] + [f'ttp_{i}' for i in range(len(mlb_ttp.classes_))])

y = df['enc_ttp'].map(lambda x: x[i])

selector = BFSThres(model, scoring='average_precision', cv = fake_splitter(df), thresh=0.01, metric_sign=1)
selector.fit(X, y, verbose=False)

filt_cols = np.array(selector.mask)
filt_cols = X.columns[filt_cols]

In [ ]:
X_tr = pd.DataFrame(df.query('split=="tr"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
             columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
y_tr = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])

X_val = pd.DataFrame(df.query('split=="val"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
             columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i])

X_ts = pd.DataFrame(df.query('split=="ts"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
             columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
y_ts = df.query('split=="ts"')['enc_ttp'].map(lambda x: x[i])

In [ ]:
N = len(filt_cols)
toolbox = get_toolbox(N=N)
toolbox.register("evaluate", evalSymbReg, points=X_tr[filt_cols].values.tolist(), y_true=y_tr)

In [ ]:
%%time
from deap import tools
from deap import algorithms

hof = tools.HallOfFame(1)

population, logbook = algorithms.eaSimple(toolbox.population(n=500), toolbox, 
                                          cxpb=0.5, mutpb=0.2, ngen=500, halloffame=hof, verbose=False)

In [ ]:
f = toolbox.compile(hof[0])

In [ ]:
from src.meta import BFSThres, fake_splitter, get_toolbox, get_gen_proba, evalSymbReg

y_tr_proba = get_gen_proba(f, X_tr, filt_cols)
y_val_proba = get_gen_proba(f, X_val, filt_cols)
y_ts_proba = get_gen_proba(f, X_ts, filt_cols)


average_precision_score(y_val, y_val_proba), average_precision_score(y_ts, y_ts_proba)

# Отбор признаков и обучение мета

In [ ]:
%%time
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from src.meta import BFSThres, fake_splitter
from src.funcs import get_pred_thresh

from sklearn.metrics import average_precision_score, f1_score

model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
                            class_weight='balanced', penalty=None)

# model = DecisionTreeClassifier(class_weight='balanced', max_depth=2, random_state=conf['seed'])
# model = GaussianNB(priors=[0.5,.5])
# model = RandomForestClassifier(n_estimators=100, max_depth=2)

average_precision_score_l = []
average_precision_score_bert_l = []
f1_score_l = []
f1_score_bert_l = []

average_precision_score_val_l = []
average_precision_score_val_bert_l = []
f1_score_val_l = []
f1_score_bert_val_l = []

for i in range(len(mlb_ttp.classes_)):
    X = df.apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1)

    X = pd.DataFrame(X.values.tolist(), columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    
    y = df['enc_ttp'].map(lambda x: x[i])
    
    selector = BFSThres(model, scoring='average_precision', cv = fake_splitter(df), thresh=0.05, metric_sign=1)
    selector.fit(X, y, verbose=True)
    
    filt_cols = np.array(selector.mask)
    filt_cols = X.columns[filt_cols]

    
    X_tr = pd.DataFrame(df.query('split=="tr"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_tr = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])
    
    X_val = pd.DataFrame(df.query('split=="val"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i])
    
    X_ts = pd.DataFrame(df.query('split=="ts"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
                 columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
    y_ts = df.query('split=="ts"')['enc_ttp'].map(lambda x: x[i])

    # model.fit(X_tr[filt_cols], y_tr)
    model.fit(X_val[filt_cols], y_val)
    
    average_precision_score_l.append(average_precision_score(y_ts, model.predict_proba(X_ts[filt_cols])[:,1]))
    average_precision_score_val_l.append(average_precision_score(y_val, model.predict_proba(X_val[filt_cols])[:,1]))
    
    _, thresh_i = get_pred_thresh(y_true = y_tr, proba=model.predict_proba(X_tr[filt_cols])[:,1], opt_metric=conf['train_eval_model']['opt_metric'],
                               thresh_space_l=np.arange(0.001, 1, 0.002))


    f1_score_l.append(f1_score(y_ts, (model.predict_proba(X_ts[filt_cols])[:,1]>thresh_i)))
    f1_score_val_l.append(f1_score(y_val, (model.predict_proba(X_val[filt_cols])[:,1]>thresh_i)))
    
    average_precision_score_bert_l.append(average_precision_score(y_ts, df.query('split=="ts"')['proba_ttp'].map(lambda x: x[i])))
    average_precision_score_val_bert_l.append(average_precision_score(y_val, df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])))
    
    f1_score_bert_l.append(f1_score(y_ts, df.query('split=="ts"')['pred_ttp'].map(lambda x: x[i])))
    f1_score_bert_val_l.append(f1_score(y_val, df.query('split=="val"')['pred_ttp'].map(lambda x: x[i])))
    


In [ ]:
np.mean(average_precision_score_val_l), np.mean(average_precision_score_val_bert_l), np.mean(f1_score_val_l), np.mean(f1_score_bert_val_l)

In [ ]:
np.mean(average_precision_score_l), np.mean(average_precision_score_bert_l), np.mean(f1_score_l), np.mean(f1_score_bert_l)

## ручной кейс

In [ ]:
from sklearn.linear_model import LogisticRegression
from src.meta import BFSThres, fake_splitter

model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
                            class_weight='balanced', penalty=None)

# for i in range(len(mlb_ttp.classes_)):

i = 36

# X =  df.apply(lambda x: x['proba_labels'] + x['proba_ttp'], axis=1)
X = df.apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1)

X = pd.DataFrame(X.values.tolist(), columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
# X = pd.DataFrame(df['feat'].values.tolist(), columns=[f'tak_{i}' for i in range(14)] + [f'ttp_{i}' for i in range(len(mlb_ttp.classes_))])

y = df['enc_ttp'].map(lambda x: x[i])

selector = BFSThres(model, scoring='average_precision', cv = fake_splitter(df), thresh=0.01, metric_sign=1)
selector.fit(X, y, verbose=True)

filt_cols = np.array(selector.mask)
filt_cols = X.columns[filt_cols]

## проверка качества

In [ ]:
X_tr = pd.DataFrame(df.query('split=="tr"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
             columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
y_tr = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])

X_val = pd.DataFrame(df.query('split=="val"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
             columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
y_val = df.query('split=="val"')['enc_ttp'].map(lambda x: x[i])

X_ts = pd.DataFrame(df.query('split=="ts"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1).values.tolist(), 
             columns=[f'tak_{i}' for i in range(len(mlb.classes_))] + ['ttp_i'])
y_ts = df.query('split=="ts"')['enc_ttp'].map(lambda x: x[i])





In [ ]:
from sklearn.metrics import average_precision_score, f1_score
model.fit(X_tr[filt_cols], y_tr)
(average_precision_score(y_tr, model.predict_proba(X_tr[filt_cols])[:,1]),
average_precision_score(y_val, model.predict_proba(X_val[filt_cols])[:,1]), 
average_precision_score(y_ts, model.predict_proba(X_ts[filt_cols])[:,1]))

In [ ]:
thresh_ttp_l[i]

In [ ]:
from src.funcs import get_pred_thresh

_, thresh_i = get_pred_thresh(y_true = y_tr, proba=model.predict_proba(X_tr[filt_cols])[:,1], opt_metric=conf['train_eval_model']['opt_metric'],
                           thresh_space_l=np.arange(0.001, 1, 0.002))


In [ ]:
from sklearn.metrics import average_precision_score, f1_score
# model.fit(X_tr[filt_cols], y_tr)
(f1_score(y_tr, (model.predict_proba(X_tr[filt_cols])[:,1]>thresh_i).astype(int)),
f1_score(y_val, (model.predict_proba(X_val[filt_cols])[:,1]>thresh_i)), 
f1_score(y_ts, (model.predict_proba(X_ts[filt_cols])[:,1]>thresh_i)))

### качество берта

In [ ]:
from sklearn.metrics import average_precision_score
(average_precision_score(y_tr, df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])),
average_precision_score(y_val, df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])), 
average_precision_score(y_ts, df.query('split=="ts"')['proba_ttp'].map(lambda x: x[i])))

In [ ]:
from sklearn.metrics import average_precision_score
(f1_score(y_tr, df.query('split=="tr"')['pred_ttp'].map(lambda x: x[i])),
f1_score(y_val, df.query('split=="val"')['pred_ttp'].map(lambda x: x[i])), 
f1_score(y_ts, df.query('split=="ts"')['pred_ttp'].map(lambda x: x[i])))

# Обучение мета с выделением

# дерево, мультибайес

In [ ]:
pr_auc_l = []
tr_pr_auc_l = []
tak_col = 'pred_labels'
split = 'tr'
# tak_col = 'pred_labels'
min_corr_thresh = 0.8
for i in range(len(mlb_ttp.classes_)):
    
    # feat_i = df.query('split=="tr"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1)
    
    corr_ind_l = pd.DataFrame(df.query('split=="tr"')['proba_labels'].values.tolist())\
        .assign(ttp = df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])).corr()\
    .loc['ttp'].abs().loc[lambda x: (x>min_corr_thresh) & (x<1)].index.tolist()
    
    if len(corr_ind_l)>0:
        print(f'For i=={i} have corr taktics - {corr_ind_l}')
        
        feat_i = df.query('split==@split').apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1)
    else:
        feat_i = df.query('split==@split').apply(lambda x: [x['proba_ttp'][i]], axis=1)

    y_i = df.query('split==@split')['enc_ttp'].map(lambda x: x[i])
    
    model = DecisionTreeClassifier(max_depth=2, random_state=conf['seed'], class_weight='balanced')\
    .fit(np.array(feat_i.values.tolist()), y_i)
    # model = GaussianNB(priors=[0.5,.5])\
    #         .fit(np.array(feat_i.values.tolist()), y_i)
    # model = CalibratedClassifierCV(LinearSVC(random_state=conf['seed'], class_weight='balanced'), 
    #                                                            cv=3, method='sigmoid').fit(np.array(feat_i.values.tolist()), y_i)

    # model = make_pipeline(RobustScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)).fit(np.array(feat_i.values.tolist()), y_i)

    # model = HistGradientBoostingClassifier(random_state=conf['seed'], class_weight='balanced').fit(np.array(feat_i.values.tolist()), y_i)
    
    # y_proba = df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])
    y_proba = model.predict_proba(np.array(
        df.query('split=="val"').apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1).values.tolist()))[:, 1]
    y_tr_proba = model.predict_proba(np.array(
        df.query('split=="tr"').apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1).values.tolist()))[:, 1]

    pr_auc_l.append(average_precision_score(df.query('split=="val"')['enc_ttp'].map(lambda x: x[i]), y_proba))
    tr_pr_auc_l.append(average_precision_score(df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i]), y_tr_proba))


In [ ]:
np.mean(pr_auc_l), np.mean(tr_pr_auc_l)

## в лоб

In [ ]:
df.query('split=="tr"')[df.query('split=="tr"')['ttp'].str.len()==8]

In [ ]:
from sklearn.decomposition import PCA

In [ ]:


model = make_pipeline(RobustScaler(), PCA(n_components=10), OneVsRestClassifier(LogisticRegression(class_weight='balanced', penalty=None, 
                                                                             max_iter=10000, random_state=conf['seed'])))

# model = LinearDiscriminantAnalysis(n_components=4)
# model.fit(np.array(df.query('split=="tr"')['feat'].values.tolist()), df.query('split=="tr"')['ttp'].values.tolist())



model.fit(np.array(df.query('split=="tr"')['feat'].values.tolist()), df.query('split=="tr"')['enc_ttp'].values.tolist())





In [ ]:
from sklearn.metrics import (log_loss, roc_auc_score, average_precision_score, f1_score, 
                            precision_recall_fscore_support, confusion_matrix)

Y_val_proba = model.predict_proba(df.query('split=="val"')['feat'].values.tolist())
Y_val = np.array(df.query('split=="val"')['enc_ttp'].values.tolist())


roc_auc, _ = metric_multi(Y_val, Y_val_proba, roc_auc_score)
logloss, _ = metric_multi(Y_val, Y_val_proba, log_loss, labels=[0,1])
pr_auc, _ = metric_multi(Y_val, Y_val_proba, average_precision_score)
roc_auc, pr_auc, logloss

In [ ]:

Y_tr_proba = model.predict_proba(df.query('split=="tr"')['feat'].values.tolist())
Y_tr = np.array(df.query('split=="tr"')['enc_ttp'].values.tolist())


roc_auc, _ = metric_multi(Y_tr, Y_tr_proba, roc_auc_score)
logloss, _ = metric_multi(Y_tr, Y_tr_proba, log_loss, labels=[0,1])
pr_auc, _ = metric_multi(Y_tr, Y_tr_proba, average_precision_score)
roc_auc, pr_auc, logloss

# Корреляция и взаимосвязи

## находим близкие тактики

In [ ]:
min_corr_thresh = 0.6
for i in range((len(mlb_ttp.classes_))):
    corr_ind_l = pd.DataFrame(df.query('split=="tr"')['proba_labels'].values.tolist())\
        .assign(ttp = df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])).corr()\
    .loc['ttp'].abs().loc[lambda x: (x>min_corr_thresh) & (x<1)].index.tolist()
    if len(corr_ind_l)>0:
        print(f'For i=={i} have corr taktics - {corr_ind_l}')

## разбираем кейс i=36

In [ ]:
min_corr_thresh = 0.6
i = 36
corr_ind_l = pd.DataFrame(df.query('split=="tr"')['proba_labels'].values.tolist())\
        .assign(ttp = df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])).corr()\
    .loc['ttp'].abs().loc[lambda x: (x>min_corr_thresh) & (x<1)].index.tolist()
corr_ind_l

In [ ]:
tak = mlb.classes_[13]
tak

In [ ]:
ttp = mlb_ttp.classes_[36]
ttp

### как часто встречается атака вместе с тактикой

In [ ]:
sel = data_ttp['ttp'].map(lambda x: ttp in x)

data_ttp.loc[sel, ['labels']].explode('labels')['labels'].unique()

In [ ]:
# exfiltraction есть везде
data_ttp.loc[sel & data_ttp['labels'].map(lambda x: tak not in x), ['labels']]

In [ ]:
data_ttp.loc[sel & data_ttp['labels'].map(lambda x: 'collection' in x), ['labels']]

In [ ]:
data_ttp[sel ].explode('labels')['labels'].value_counts()

### есть ли кейс, чтобы техника не предсказана правильно, а атака exfiltration 

In [ ]:
# есть кейсы, где мы неправильно предсказали технику T1041, но тактику правильно. 
# Получается для этих кейсов наш алгоритм должен был улучшить 

sel = (df['labels'].map(lambda x: tak in x)) & (df['ttp'].map(lambda x: ttp in x)) \
        & (df['pred_str_ttp'].map(lambda x: not ttp in x)) & (df['pred_str_labels'].map(lambda x: tak in x)) 
df[sel].head(2)

### формируем фичи и пытаемся предсказать

In [ ]:
mlb_ttp.classes_[i], mlb.classes_[13]

In [ ]:
df

In [ ]:
# tak_col = 'pred_labels'
tak_col = 'proba_labels'
df['feature'] = df.apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1)

In [ ]:
df.loc[sel, ['labels', 'ttp', 'pred_str_labels', 'pred_str_ttp', 'feature']].head(2)

In [ ]:
thresh_ttp_l[i]

In [ ]:
df[(df['labels'].map(lambda x: tak in x)) & (df['ttp'].map(lambda x: ttp in x)) \
        & (df['pred_str_ttp'].map(lambda x: ttp in x)) & (df['pred_str_labels'].map(lambda x: tak in x))].head(2)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(fit_intercept=False, random_state=conf['seed'], 
                                class_weight='balanced', penalty=None,
                          ).fit(df.query('split=="tr"')['feature'].values.tolist(), 
                                 df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i]))


In [ ]:
model.coef_

<div class='alert alert-info'> 
    При высокой корреляции нельзя применять логистическую регрессию
    
</div>

# логрег

In [ ]:
from sklearn.decomposition import PCA, NMF

In [ ]:
pr_auc_l = []

min_corr_thresh = 0.1
tak_col = 'pred_labels'

probas_l = []
tr_pr_auc_l = []
for i in range(len(mlb_ttp.classes_)):
    
    # feat_i = df.query('split=="tr"').apply(lambda x: x['proba_labels'] + [x['proba_ttp'][i]], axis=1)
    
    corr_ind_l = pd.DataFrame(df.query('split=="tr"')['proba_labels'].values.tolist())\
        .assign(ttp = df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])).corr()\
    .loc['ttp'].abs().loc[lambda x: (x>min_corr_thresh) & (x<1)].index.tolist()
    
    if len(corr_ind_l)>0:
        print(f'For i=={i} have corr taktics - {corr_ind_l}')
        feat_i = df.query('split=="tr"').apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1)
    else:
        feat_i = df.query('split=="tr"').apply(lambda x: [x['proba_ttp'][i]], axis=1)

    y_i = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])
    
    model = make_pipeline(NMF(n_components=1), LogisticRegression(penalty='l2', C=15, fit_intercept=False, random_state=conf['seed'], 
                                                             class_weight='balanced'))\
            .fit(np.array(feat_i.values.tolist()), y_i)
    
    # y_proba = df.query('split=="val"')['proba_ttp'].map(lambda x: x[i])
    y_proba = model.predict_proba(np.array(
        df.query('split=="val"').apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1).values.tolist()))[:, 1]
    y_tr_proba = model.predict_proba(np.array(
        df.query('split=="tr"').apply(lambda x: np.array(x[tak_col])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1).values.tolist()))[:, 1]
    probas_l.append(y_proba)
    # tr_probas_l.append(y_tr_proba)
    
    pr_auc_l.append(average_precision_score(df.query('split=="val"')['enc_ttp'].map(lambda x: x[i]), y_proba))
    tr_pr_auc_l.append(average_precision_score(df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i]), y_tr_proba))

In [ ]:
np.mean(pr_auc_l), np.mean(tr_pr_auc_l)

In [ ]:
probas = np.column_stack(probas_l)

In [ ]:
thresh_l = get_opt_thresh(y_true=np.array(df.query('split=="val"')['enc_ttp'].values.tolist()), 
               probas=probas, mlb=mlb_ttp, opt_metric='f1', thresh_space_l=np.arange(0.005, 1, 0.005))

In [ ]:
er_df = df.query('split=="val"').copy()
er_df['proba_new'] = probas.tolist()

In [ ]:
er_df['pred_new'] = er_df['proba_new'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_l)])

In [ ]:
p_val_micro, r_val_micro, f1_val_micro, sup = precision_recall_fscore_support(np.array(er_df['enc_ttp'].values.tolist()), 
                                                    np.array(er_df['pred_new'].values.tolist()), average='micro')
p_val_macro, r_val_macro, f1_val_macro, sup = precision_recall_fscore_support(np.array(er_df['enc_ttp'].values.tolist()), 
                                                    np.array(er_df['pred_new'].values.tolist()), average='macro')

f1_val_micro, f1_val_macro

### пристальнее

In [ ]:
min_corr_thresh = 0.9
probas_l = []
i=3
corr_ind_l = pd.DataFrame(df.query('split=="tr"')['proba_labels'].values.tolist())\
    .assign(ttp = df.query('split=="tr"')['proba_ttp'].map(lambda x: x[i])).corr()\
.loc['ttp'].abs().loc[lambda x: (x>0.3) & (x<1)].index.tolist()

corr_ind_l

In [ ]:
feat_i = df.query('split=="tr"').apply(lambda x: [x['proba_ttp'][i]], axis=1)
    
feat_i

In [ ]:
y_i = df.query('split=="tr"')['enc_ttp'].map(lambda x: x[i])

model = LogisticRegression(penalty=None, fit_intercept=False, random_state=conf['seed'], class_weight='balanced')\
        .fit(np.array(feat_i.values.tolist()), y_i)

In [ ]:
np.array(feat_i.values.tolist())[(y_i==0)]

In [ ]:
feat_i.map(lambda x: x[0])[(y_i==1)]

In [ ]:
feat_i.map(lambda x: x[0])[(y_i==1)].describe()

In [ ]:
feat_i.map(lambda x: x[0])[(y_i==0)].describe()

In [ ]:
y_proba = model.predict_proba(np.array(
    df.query('split=="val"').apply(lambda x: np.array(x['proba_labels'])[corr_ind_l].tolist() + [x['proba_ttp'][i]], axis=1).values.tolist()))[:, 1]

## свое предсказание для тактик

# train f1

In [ ]:
conf = YAML().load(open('params.yaml'))
set_seed(conf['seed'])

conf_dop = YAML().load(open('dvc_pipes/bert/params_bert.yaml'))

In [ ]:
target_col='labels'

thresh_space_l=[]

In [ ]:
from sklearn.metrics import (average_precision_score, log_loss, confusion_matrix,
                            precision_recall_fscore_support, f1_score)

import matplotlib.pyplot as plt
import seaborn as sns

import os

import pandas as pd
import numpy as np
import joblib

from collections import defaultdict
from itertools import chain


import json

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ExponentialLR, MultiStepLR
from torch.utils.data import DataLoader, Dataset


from transformers import BertTokenizer, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModelForMaskedLM, BertConfig, AutoModel
from transformers import DataCollatorWithPadding
from transformers import RobertaTokenizer, RobertaModel

DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

import sys
sys.path.append('.')

from src.funcs import metric_multi
from src.funcs import get_opt_thresh, get_preds
from src.funcs import get_conf_df


class TextDFDataset(Dataset):

    def __init__(self, df, tokenizer, tokenizer_opts):

      self.df = df
      self.tokenizer = tokenizer
      self.tokenizer_opts = tokenizer_opts 
        
    def __getitem__(self, idx):
      # import pdb;pdb.set_trace()
      tok_d = self.tokenizer(self.df.loc[self.df.index[idx], 'sentence'], **self.tokenizer_opts)

      return {**{k:v.flatten() for k,v in tok_d.items()}, **{'target':torch.tensor(self.df.loc[self.df.index[idx], 'target'], dtype=torch.float)}}

    def __len__(self):

      return self.df.shape[0]


class TextModelClass(torch.nn.Module):

    def __init__(self, bert_model, mode, classnum, dropout_ratio):

        super().__init__()
        self.bert = bert_model
        self.mode = mode
        self.lin = torch.nn.Linear(768, 768)
        self.drop_out = torch.nn.Dropout(dropout_ratio)
        self.lin_out = torch.nn.Linear(768, classnum)

    def forward(self, X):

        out = self.bert(**{k: v.to(DEVICE) for k, v in X.items() if k!='target'})
        if self.mode=='cls':
            out = out.last_hidden_state[:,0,:]
        elif self.mode=='pooler':
            out = out.pooler_output
        out = self.lin(out)
        out = torch.nn.ReLU()(out)
        out = self.drop_out(out)
        out = self.lin_out(out)

        return out



In [ ]:
TRAIN_BATCH_SIZE = conf_dop['nn']['batch_size']
VALID_BATCH_SIZE = conf_dop['nn']['batch_size']

MAX_SEQ_LENGTH = conf_dop['nn']['maxlen']

DROPOUT_RATIO = conf_dop['nn_bert']['drop_ratio']

mode = conf_dop['nn_bert']['mode'] # pooler

LEARNING_RATE = conf_dop['nn']['learning_rate']
EPOCH_NUM = conf_dop['nn']['epoch_num']

exp_gamma = conf_dop['nn']['exp_gamma'] 
milestone_gamma = conf_dop['nn']['milestone_gamma']
milestone_l = conf_dop['nn']['milestone_l'] 
l2 = conf_dop['nn']['l2']

bert_type = conf_dop['nn_bert']['bert_type']

if target_col=='ttp':
    mlb = joblib.load(conf['prep_text']['ttp_mlb_fn'])
else:
    mlb = joblib.load(conf['prep_text']['mlb_fn'])

data = pd.read_csv(conf['feat_gen']['data_fn'])

data['target'] = data['target'].map(lambda x: eval(x))
data[target_col] = data[target_col].map(lambda x: eval(x))
tr_idx = data.query('split=="tr"').index
val_idx = data.query('split=="val"').index
ts_idx = data.query('split=="ts"').index


if bert_type == 'secbert_plus':
    
    # checkpoint = "ehsanaghaei/SecureBERT_Plus"
    checkpoint = 'data/external/models/SecureBERT_Plus/snapshots/4c48ccdb8d2019f179b07dfa27656c655394d78e'
    tokenizer = RobertaTokenizer.from_pretrained(checkpoint)
    model = RobertaModel.from_pretrained(checkpoint, output_hidden_states=True)
    tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}
    bert_model= model
    
elif bert_type == 'secbert':

    # checkpoint = "jackaduma/SecBERT"
    checkpoint = 'data/external/models/SecBERT/snapshots/7c603df5bc4c5ba9c731bcc2ea0ab2db36e104cb'
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    tokenizer_opts = {'max_length':MAX_SEQ_LENGTH, 'return_tensors':"pt", 'padding':True, 'truncation':True, 'add_special_tokens':True}
    config = BertConfig.from_pretrained(checkpoint, output_hidden_states=True)
    bert_model = AutoModel.from_pretrained(checkpoint, config=config).base_model

elif bert_type == 'scibert':
    # checkpoint = 'allenai/scibert_scivocab_uncased'
    checkpoint = 'data/external/models/scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1'
    tokenizer = BertTokenizer.from_pretrained(checkpoint, max_length=512)
    model = BertForSequenceClassification.from_pretrained('data/external/models/scibert_scivocab_uncased/scibert_multi_label_model')
    bert_model= model.bert
    tokenizer_opts = {'return_tensors':"pt", 'truncation':True,
                      'padding':'max_length', 'max_length':MAX_SEQ_LENGTH}

tr_ds = TextDFDataset(data.query('split=="tr"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
val_ds = TextDFDataset(data.query('split=="val"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)
ts_ds = TextDFDataset(data.query('split=="ts"').reset_index(drop=True), tokenizer=tokenizer, tokenizer_opts=tokenizer_opts)

tr_ld = DataLoader(tr_ds, batch_size = TRAIN_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
val_ld = DataLoader(val_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))
ts_ld = DataLoader(ts_ds, batch_size = VALID_BATCH_SIZE, shuffle = False, collate_fn = DataCollatorWithPadding(tokenizer=tokenizer))

model = TextModelClass(bert_model, mode=mode, classnum=len(mlb.classes_), dropout_ratio=DROPOUT_RATIO)
model = model.to(DEVICE)

# Заморозим все слои берта
for param in model.bert.parameters():
    param.requires_grad = False

loss_fn = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE, weight_decay=l2)
# optimizer = torch.optim.RMSprop(model.parameters(), lr=LEARNING_RATE)

scheduler1 = ExponentialLR(optimizer, gamma=conf_dop['nn']['exp_gamma'] )
scheduler2 = MultiStepLR(optimizer, milestones=conf_dop['nn']['milestone_l'], gamma=conf_dop['nn']['milestone_gamma'])

val_iter_num = 1
refresh_cache_iter = 10

loss_d = {}

name = os.path.basename(conf_dop['nn_bert']['model_fn'])
dirname = os.path.dirname(conf_dop['nn_bert']['model_fn'])

if conf_dop['nn_bert']['from_existing']:

    model = torch.load(f'{dirname}/{bert_type}_{name}')
    # saving old one

    os.rename(f'{dirname}/{bert_type}_{name}', f'{dirname}/prev_{bert_type}_{name}')


In [ ]:
for epoch in range(1, 2):
    loss_tr_l = []
    model.train()
    res_d = defaultdict(list)
    tr_batch_num = len(tr_ld)
    tr_loss_epoch = 0
    
    for batch_tr in tr_ld:
        out = model(batch_tr)
        # import pdb;pdb.set_trace()
        optimizer.zero_grad()

        loss = loss_fn(out, batch_tr['target'].to(DEVICE))
        loss.backward()
        optimizer.step()
        tr_loss_epoch = tr_loss_epoch + loss.item()
        with torch.no_grad():
            model.eval()
            res_d['tr_target'].append(batch_tr['target'].numpy()) 
            res_d['tr_pred'].append(out.sigmoid().cpu().numpy()) 

    res_d['tr_target'] = list(chain(*res_d['tr_target']))
    res_d['tr_pred'] = list(chain(*res_d['tr_pred']))
    
    scheduler1.step()
    scheduler2.step()
    
    if epoch%val_iter_num==0:
        model.eval()
        val_batch_num = len(val_ld)
        val_loss_epoch = 0
        pr_auc = 0
        with torch.no_grad():
            for batch_val in val_ld:
                pred = model(batch_val)
                val_loss = loss_fn(pred, batch_val['target'].to(DEVICE))
                val_loss_epoch = val_loss_epoch+val_loss.item()
                sigm_preds = pred.sigmoid().cpu()

                pr_auc = pr_auc + metric_multi(batch_val['target'].numpy(), sigm_preds.numpy(), average_precision_score)[0]

                res_d['target'].append(batch_val['target'].numpy())
                res_d['pred'].append(sigm_preds.numpy())

            res_d['target'] = list(chain(*res_d['target']))
            res_d['pred'] = list(chain(*res_d['pred']))


    loss_d[epoch] = {'log_loss_tr_batch':tr_loss_epoch/tr_batch_num,
                      'log_loss_val_batch':val_loss_epoch/val_batch_num,
                    'pr_auc_batch':pr_auc/val_batch_num,
                     'log_loss_val':metric_multi(np.array(res_d['target']), np.array(res_d['pred']), log_loss)[0],
                    'log_loss_tr':metric_multi(np.array(res_d['tr_target']), np.array(res_d['tr_pred']), log_loss)[0],
                     'pr_auc_val':metric_multi(np.array(res_d['target']), np.array(res_d['pred']), average_precision_score)[0],
                    'pr_auc_tr':metric_multi(np.array(res_d['tr_target']), np.array(res_d['tr_pred']), average_precision_score)[0]}
    print(f'epoch num - {epoch}')

In [ ]:
thresh_col = 'y_p'

In [ ]:
res_tr_df = pd.DataFrame({'y_proba':np.array(res_d['tr_pred']).tolist(), 'y':np.array(res_d['tr_target']).tolist()})

res_tr_df[thresh_col] = res_tr_df['y_proba'].map(lambda x: [int(val>=thresh) for val, thresh in zip(x, thresh_l)])

    
p_tr_micro, r_tr_micro, f1_tr_micro, sup = precision_recall_fscore_support(np.array(res_tr_df['y'].values.tolist()), 
                                                    np.array(res_tr_df[thresh_col].values.tolist()), average='micro')
p_tr_macro, r_tr_macro, f1_tr_macro, sup = precision_recall_fscore_support(np.array(res_tr_df['y'].values.tolist()), 
                                                    np.array(res_tr_df[thresh_col].values.tolist()), average='macro')

f1_tr_micro, f1_tr_macro